In [42]:
import pandas as pd

In [43]:
data=pd.read_csv("HousePricePrediction.csv")

In [44]:
data

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2914,160,RM,1936,Inside,Twnhs,7,1970,1970,CemntBd,0.0,546.0,NaN
2915,2915,160,RM,1894,Inside,TwnhsE,5,1970,1970,CemntBd,0.0,546.0,NaN
2916,2916,20,RL,20000,Inside,1Fam,7,1960,1996,VinylSd,0.0,1224.0,NaN
2917,2917,85,RL,10441,Inside,1Fam,5,1992,1992,HdBoard,0.0,912.0,NaN


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            2919 non-null   int64  
 1   MSSubClass    2919 non-null   int64  
 2   MSZoning      2915 non-null   object 
 3   LotArea       2919 non-null   int64  
 4   LotConfig     2919 non-null   object 
 5   BldgType      2919 non-null   object 
 6   OverallCond   2919 non-null   int64  
 7   YearBuilt     2919 non-null   int64  
 8   YearRemodAdd  2919 non-null   int64  
 9   Exterior1st   2918 non-null   object 
 10  BsmtFinSF2    2918 non-null   float64
 11  TotalBsmtSF   2918 non-null   float64
 12  SalePrice     1460 non-null   float64
dtypes: float64(3), int64(6), object(4)
memory usage: 296.6+ KB


In [43]:
# means it contains test data also whose sales value is null
data["SalePrice"].isnull().sum()

np.int64(1459)

**Data Preprocessing**

1. Handling missing values (filled with 0 here)

In [44]:

# Wrong approach as for categorical columns fill the missing value with "Missing" not 0 as during one hot encdding data mismatch problem will be there
# data["MSZoning"]=data["MSZoning"].fillna(0)
# data["Exterior1st"]=data["Exterior1st"].fillna(0)

print(data["MSZoning"].unique())

['RL' 'RM' 'C (all)' 'FV' 'RH' nan]


In [45]:
print(data["LotConfig"].unique())

['Inside' 'FR2' 'Corner' 'CulDSac' 'FR3']


In [46]:
print(data["BldgType"].unique())

['1Fam' '2fmCon' 'Duplex' 'TwnhsE' 'Twnhs']


In [47]:
print(data["Exterior1st"].unique())

['VinylSd' 'MetalSd' 'Wd Sdng' 'HdBoard' 'BrkFace' 'WdShing' 'CemntBd'
 'Plywood' 'AsbShng' 'Stucco' 'BrkComm' 'AsphShn' 'Stone' 'ImStucc'
 'CBlock' nan]


In [48]:
# For non categorical column fill with median or mean value it is better then 0
# data["BsmtFinSF2"]=data["BsmtFinSF2"].fillna(0)
# data["TotalBsmtSF"]=data["TotalBsmtSF"].fillna(0)

In [50]:
categorical_cols=data.select_dtypes(include="object").columns
non_categorical_cols=data.select_dtypes(exclude="object").columns.drop("SalePrice")

In [51]:
data[categorical_cols]=data[categorical_cols].fillna("Missing")
data[non_categorical_cols]=data[non_categorical_cols].fillna(data[non_categorical_cols].median())

2. One hot encoding 

In [52]:
from sklearn.preprocessing import OneHotEncoder

In [53]:
encoder=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
encoded = encoder.fit_transform(data[categorical_cols])


# Convert encoded array to dataframe
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=data.index
)

# Drop original categorical columns
data_numeric = data.drop(columns=categorical_cols)

# Concatenate encoded columns
data_final = pd.concat([data_numeric, encoded_df], axis=1)
print(data_final)


        Id  MSSubClass  LotArea  OverallCond  YearBuilt  YearRemodAdd  \
0        0          60     8450            5       2003          2003   
1        1          20     9600            8       1976          1976   
2        2          60    11250            5       2001          2002   
3        3          70     9550            5       1915          1970   
4        4          60    14260            5       2000          2000   
...    ...         ...      ...          ...        ...           ...   
2914  2914         160     1936            7       1970          1970   
2915  2915         160     1894            5       1970          1970   
2916  2916          20    20000            7       1960          1996   
2917  2917          85    10441            5       1992          1992   
2918  2918          60     9627            5       1993          1994   

      BsmtFinSF2  TotalBsmtSF  SalePrice  MSZoning_C (all)  ...  \
0            0.0        856.0   208500.0               0

In [54]:
# Saving the preprocessed file
data_final.to_csv("processed_house_price_prediction.csv", index=False)

**ADDING THE TEST DATA ALSO (SETTING ITS SALE PRICE VALUE AS MEAN)**

In [45]:
# setting the value of saleprice first

data["SalePrice"]=data["SalePrice"].fillna(data["SalePrice"].mean())

In [46]:
data

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.00000
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.00000
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.00000
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.00000
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2914,160,RM,1936,Inside,Twnhs,7,1970,1970,CemntBd,0.0,546.0,180921.19589
2915,2915,160,RM,1894,Inside,TwnhsE,5,1970,1970,CemntBd,0.0,546.0,180921.19589
2916,2916,20,RL,20000,Inside,1Fam,7,1960,1996,VinylSd,0.0,1224.0,180921.19589
2917,2917,85,RL,10441,Inside,1Fam,5,1992,1992,HdBoard,0.0,912.0,180921.19589


In [47]:
data.isnull().sum()

Id              0
MSSubClass      0
MSZoning        4
LotArea         0
LotConfig       0
BldgType        0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
Exterior1st     1
BsmtFinSF2      1
TotalBsmtSF     1
SalePrice       0
dtype: int64

In [48]:
data=data.dropna()

In [49]:
data.isnull().sum()

Id              0
MSSubClass      0
MSZoning        0
LotArea         0
LotConfig       0
BldgType        0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
Exterior1st     0
BsmtFinSF2      0
TotalBsmtSF     0
SalePrice       0
dtype: int64

In [50]:
data

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.00000
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.00000
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.00000
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.00000
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2914,160,RM,1936,Inside,Twnhs,7,1970,1970,CemntBd,0.0,546.0,180921.19589
2915,2915,160,RM,1894,Inside,TwnhsE,5,1970,1970,CemntBd,0.0,546.0,180921.19589
2916,2916,20,RL,20000,Inside,1Fam,7,1960,1996,VinylSd,0.0,1224.0,180921.19589
2917,2917,85,RL,10441,Inside,1Fam,5,1992,1992,HdBoard,0.0,912.0,180921.19589


In [51]:
# doing one hot encoding
categorical_cols=data.select_dtypes(include="object").columns
print(categorical_cols)

Index(['MSZoning', 'LotConfig', 'BldgType', 'Exterior1st'], dtype='object')


In [52]:
data=pd.get_dummies(data,columns=categorical_cols,drop_first=True,dtype=int)
data

,Id,MSSubClass,LotArea,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF2,TotalBsmtSF,SalePrice,MSZoning_FV,...,Exterior1st_CemntBd,Exterior1st_HdBoard,Exterior1st_ImStucc,Exterior1st_MetalSd,Exterior1st_Plywood,Exterior1st_Stone,Exterior1st_Stucco,Exterior1st_VinylSd,Exterior1st_Wd Sdng,Exterior1st_WdShing
0,0,60,8450,5,2003,2003,0.0,856.0,208500.00000,0,...,0,0,0,0,0,0,0,1,0,0
1,1,20,9600,8,1976,1976,0.0,1262.0,181500.00000,0,...,0,0,0,1,0,0,0,0,0,0
2,2,60,11250,5,2001,2002,0.0,920.0,223500.00000,0,...,0,0,0,0,0,0,0,1,0,0
3,3,70,9550,5,1915,1970,0.0,756.0,140000.00000,0,...,0,0,0,0,0,0,0,0,1,0
4,4,60,14260,5,2000,2000,0.0,1145.0,250000.00000,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2914,160,1936,7,1970,1970,0.0,546.0,180921.19589,0,...,1,0,0,0,0,0,0,0,0,0
2915,2915,160,1894,5,1970,1970,0.0,546.0,180921.19589,0,...,1,0,0,0,0,0,0,0,0,0
2916,2916,20,20000,7,1960,1996,0.0,1224.0,180921.19589,0,...,0,0,0,0,0,0,0,1,0,0
2917,2917,85,10441,5,1992,1992,0.0,912.0,180921.19589,0,...,0,1,0,0,0,0,0,0,0,0


In [54]:
data.to_csv("processed_house_price_prediction.csv", index=False)